# MULCURSIO: Train Divination Control Vector (RunPod)

This notebook is designed to run on RunPod via remote Jupyter kernel.

## Prerequisites

1. SSH tunnel to RunPod: `ssh -o IdentitiesOnly=yes -i ~/.ssh/id_ed25519 -L 8888:localhost:8888 sfghgd9eq41tsn-64411155@ssh.runpod.io`
2. Jupyter Lab running on pod: `cd ~/mulcursio/notebooks && jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root`
3. Connect Cursor to remote kernel at `http://localhost:8888`

## Setup

In [ ]:
# Verify CUDA is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("WARNING: CUDA not available, will use CPU (slow!)")

In [ ]:
# Set up paths
import sys
from pathlib import Path

# Add parent directories to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root.parent / 'repeng'))  # For repeng library

print(f"Project root: {project_root}")
print(f"Python path: {sys.path[:3]}")

In [ ]:
# Install dependencies if needed
try:
    import repeng
    print("✓ repeng installed")
except ImportError:
    print("Installing repeng...")
    %pip install -e ~/repeng

try:
    import transformers
    print("✓ transformers installed")
except ImportError:
    print("Installing dependencies...")
    %pip install torch transformers accelerate numpy gguf

## 1. Load and Inspect Contrastive Pairs

In [ ]:
from src.train_vector import load_divination_pairs

# Load pairs
pairs_path = str(project_root / "data" / "divination_pairs.json")
pairs = load_divination_pairs(pairs_path)

print(f"Loaded {len(pairs)} contrastive pairs\n")

# Show a few examples
for i, (pos, neg) in enumerate(pairs[:5]):
    print(f"Pair {i+1}:")
    print(f"  (+) {pos}")
    print(f"  (-) {neg}")
    print()

## 2. Configure Training

In [ ]:
# Training configuration
CONFIG = {
    "model_name": "mistralai/Mistral-7B-Instruct-v0.1",
    "pairs_path": pairs_path,
    "output_path": str(project_root / "vectors" / "divination_v1.gguf"),
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "batch_size": 8,  # Reduce to 4 or 2 if OOM
    "method": "pca_diff",  # or "pca_center"
    "use_expansion": False,  # Set True to use suffix expansion
    "suffixes_path": None  # Set path if using expansion
}

print("Training Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 3. Train Control Vector

This will download the model (~14GB) on first run, then train for 10-20 minutes.

In [ ]:
from src.train_vector import train_divination_vector

# Train the vector
vector = train_divination_vector(
    model_name=CONFIG["model_name"],
    pairs_path=CONFIG["pairs_path"],
    suffixes_path=CONFIG["suffixes_path"],
    output_path=CONFIG["output_path"],
    device=CONFIG["device"],
    batch_size=CONFIG["batch_size"],
    method=CONFIG["method"],
    use_expansion=CONFIG["use_expansion"]
)

print("\n✓ Training complete!")
print(f"Vector saved to: {CONFIG['output_path']}")

## 4. Test the Vector

Let's test the control vector with a simple generation.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from repeng import ControlModel, ControlVector
import torch

print("Loading model for testing...")

# Load model
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    device_map=CONFIG["device"]
)
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
tokenizer.pad_token = tokenizer.eos_token

# Wrap for control
layer_ids = list(range(-5, -18, -1))
model = ControlModel(model, layer_ids)

# Load the trained vector
vector = ControlVector.import_gguf(CONFIG["output_path"])

print("✓ Model ready")

In [ ]:
# Test prompt
test_prompt = "[INST] What do you see in the patterns of existence? [/INST]"

print("Testing control vector...\n")
print("="*60)

# Generate without control
print("\nBASELINE (no control):")
print("-"*60)
model.reset()
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=100, temperature=0.8, do_sample=True)
baseline = tokenizer.decode(output[0], skip_special_tokens=True)
print(baseline.replace(test_prompt.replace("[INST] ", "").replace(" [/INST]", ""), "").strip())

print("\n" + "="*60)

# Generate with control (coefficient = 1.5)
print("\nWITH CONTROL (coeff=1.5):")
print("-"*60)
model.set_control(vector, coeff=1.5)
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=100, temperature=0.8, do_sample=True)
controlled = tokenizer.decode(output[0], skip_special_tokens=True)
print(controlled.replace(test_prompt.replace("[INST] ", "").replace(" [/INST]", ""), "").strip())

print("\n" + "="*60)

# Generate with higher control (coefficient = 2.0)
print("\nWITH HIGH CONTROL (coeff=2.0):")
print("-"*60)
model.set_control(vector, coeff=2.0)
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=100, temperature=0.8, do_sample=True)
high_controlled = tokenizer.decode(output[0], skip_special_tokens=True)
print(high_controlled.replace(test_prompt.replace("[INST] ", "").replace(" [/INST]", ""), "").strip())

print("\n" + "="*60)

## 5. Save Training Report

In [ ]:
import json
from datetime import datetime

report = {
    "timestamp": datetime.now().isoformat(),
    "config": CONFIG,
    "num_pairs": len(pairs),
    "test_results": {
        "prompt": test_prompt,
        "baseline": baseline,
        "controlled_1.5": controlled,
        "controlled_2.0": high_controlled
    },
    "device_info": {
        "cuda_available": torch.cuda.is_available(),
        "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "cuda_version": torch.version.cuda if torch.cuda.is_available() else None
    }
}

report_path = project_root / "vectors" / "training_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Training report saved to: {report_path}")
print("\n✓ Training session complete!")
print(f"\nTrained vector: {CONFIG['output_path']}")
print("\nNext steps:")
print("  1. Download vector: ./scripts/runpod_download.sh vectors")
print("  2. Run session: python scripts/run_session.py --turns 50")
print("  3. Analyze: notebooks/03_analyze_session.ipynb")

## 6. Optional: Run a Quick Test Session

Run a short backroom conversation to verify everything works.

In [ ]:
from src.backroom_engine import BackroomEngine, AgentConfig
from src.logger import ConversationLogger
from datetime import datetime

# Configure agents
agent_a = AgentConfig(
    name="Agent_A",
    control_vector_path=None,
    control_coeff=0.0
)

agent_b = AgentConfig(
    name="Agent_B",
    control_vector_path=CONFIG["output_path"],
    control_coeff=1.2
)

# Create engine (reuses model already loaded)
print("Initializing backroom engine...")
engine = BackroomEngine(
    model_name=CONFIG["model_name"],
    agent_a_config=agent_a,
    agent_b_config=agent_b,
    device=CONFIG["device"]
)

# Create logger
session_id = f"test_session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
session_dir = project_root / "sessions" / session_id
logger = ConversationLogger(session_dir, save_attention=False)

# Run short session (5 turns)
print("\nRunning test session (5 turns)...")
engine.run_session(
    logger=logger,
    seed_prompt="We are two minds meeting in the infinite backroom. What do you perceive?",
    num_turns=5,
    max_new_tokens=128,
    temperature=0.8,
    save_attention=False
)

logger.finalize({"test": True, **CONFIG})

print("\n✓ Test session complete!")
print(f"Session saved to: {session_dir}")

## Done!

The control vector is trained and tested. Download it to your local machine with:

```bash
./scripts/runpod_download.sh vectors
```